# Preparation and training

Last updated 2026-11-24 14:21 JST

Use this notebook for T2TT, speech preparation, Baseline training, teacher construction, calibration, and the three transfer conditions. Compare results in [evaluate.ipynb](evaluate.ipynb).

1. To prepare audio ZIPs, use a CPU runtime and set `PREPARE_AUDIO_PACKS=True`. `AUDIO_PACK_WORKERS` accepts 1–4 workers and defaults to 2.
2. For training, switch to GPU, set `PREPARE_AUDIO_PACKS=False`, and rerun from the top with the same settings.
3. Enable the desired training stages. You can restore a trained T2TT model from its saved run.
4. Review the top-p diagnostics, set `MASK_TOP_P`, and calibrate before running the transfer Pilot. Review the Pilot before continuing to 400k.

After a disconnect, rerun from the top with the same experiment ID, revision, and settings. CPU ZIP mode skips training cells.

Use `DATA_CONFIG` or the input overrides to specify the corpus root and manifest. Experiment IDs select output locations. To reuse a saved model, specify its original ID.

## GitHub access

For a private repository, create a fine-grained personal access token in GitHub **Settings → Developer settings → Personal access tokens → Fine-grained tokens**. Select the repository owner, choose **Only select repositories**, select `alignment-transfer-s2ut`, and grant **Contents → Read-only**. Set an expiration date. See the [GitHub token guide](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens).

In Colab, open **Secrets** (the key icon), add `GITHUB_TOKEN`, paste the token into its value, and enable **Notebook access**. The setup cell reads this secret for Git operations. Public repositories also work without a token. Successful clone/fetch and checkout in the setup cell confirm access to the requested revision.

To authenticate an additional repository, add its exact HTTPS URL to `GITHUB_AUTH_REPOS` in the settings cell and grant the same token read access to that repository. The benchmark is public and uses anonymous HTTPS by default.

After a runtime reset, use the same notebook revision, experiment settings, and Drive paths, then rerun the cells. To open the notebook after the repository becomes private, download the `.ipynb` while signed into GitHub and use **File → Upload notebook** in Colab, or open a copy saved in your Drive. Enable secret access for that notebook. The runtime token is used after the notebook opens.


In [ ]:
#@title Process logging and progress
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


"""Standard-library GitHub authentication, embedded before Colab installation."""


GITHUB_GIT_WORKER = r'''
import base64
import os
import subprocess
import sys
from urllib.parse import quote

redactions = []
for i in range(int(os.environ.get("GIT_CONFIG_COUNT", "0"))):
    value = os.environ.get(f"GIT_CONFIG_VALUE_{i}", "")
    if value.startswith("Authorization: Basic "):
        encoded = value.split()[-1]
        token = base64.b64decode(encoded).decode("utf-8").split(":", 1)[1]
        redactions.extend((value, encoded, token, quote(token, safe="")))
redactions = sorted(set(redactions), key=len, reverse=True)
process = None
try:
    command = sys.argv[1:]
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    if command[position] in {"fetch", "pull"}:
        remote = next((arg for arg in command[position + 1:] if not arg.startswith("-")), "origin")
        if "://" not in remote:
            checked = subprocess.run([*command[:position], "remote", "get-url", "--all", remote],
                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                text=True, encoding="utf-8", errors="replace")
            from urllib.parse import urlsplit
            for address in checked.stdout.splitlines():
                url = urlsplit(address)
                if url.username or url.password or url.query or url.fragment:
                    raise ValueError("unsafe stored remote")
    process = subprocess.Popen(command, stdin=subprocess.DEVNULL,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        for secret in redactions:
            line = line.replace(secret, "[REDACTED]")
        print(line, end="", flush=True)
    code = process.wait()
except Exception:
    print("Git could not complete the requested operation.", file=sys.stderr, flush=True)
    code = 1
finally:
    if process is not None:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
if code:
    print("Check the repository URL and revision. For a private repository, add "
          "GITHUB_TOKEN to Colab Secrets, enable notebook access, and grant the "
          "fine-grained token Contents: Read-only for the selected repository. "
          "Check token expiry and any organization approval.", file=sys.stderr, flush=True)
sys.exit(code)
'''


def run_github_git(command, logs, *, runner, repositories, name="git", env=None, cwd=None,
                   capture_json=False):
    """Authenticate network Git calls with URL-scoped, process-only headers."""
    import base64
    import os
    from pathlib import Path
    import re
    import sys
    from urllib.parse import urlsplit

    command = list(map(str, command))
    position = 1
    while position < len(command) and command[position] == "-C":
        position += 2
    operation = command[position] if position < len(command) else ""
    if operation not in {"clone", "fetch", "pull", "ls-remote"}:
        return runner(command, logs, name=name, env=env, cwd=cwd, capture_json=capture_json)
    if Path(command[0]).name.lower() not in {"git", "git.exe"}:
        raise ValueError("GitHub authentication requires a git command")
    for argument in command:
        if "://" in argument:
            url = urlsplit(argument)
            if url.username or url.password or url.query or url.fragment:
                raise ValueError("Use repository URLs without credentials, queries, or fragments")
    scopes = []
    for repository in repositories:
        url = urlsplit(str(repository))
        if (url.scheme != "https" or url.netloc != "github.com" or url.query or url.fragment
                or not re.fullmatch(r"/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", url.path)):
            raise ValueError("Authentication scopes must be plain HTTPS GitHub repository URLs")
        scopes.append(url.geturl())

    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Public repositories remain accessible when Secrets are unavailable.
    if token is not None and (not isinstance(token, str) or any(ord(c) < 33 or ord(c) > 126 for c in token)):
        token = None
        raise ValueError("GITHUB_TOKEN must contain a valid GitHub token without whitespace")
    git_env = dict(os.environ if env is None else env)
    for key in list(git_env):
        if (key.startswith(("GIT_TRACE", "GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))
                or key in {"GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS", "GIT_CURL_VERBOSE", "GITHUB_TOKEN", "GH_TOKEN"}):
            git_env.pop(key)
    git_env.update(GIT_TERMINAL_PROMPT="0", GCM_INTERACTIVE="Never", GIT_ASKPASS="", GIT_TRACE_REDACT="1")
    settings = [("credential.helper", ""), ("http.extraHeader", ""), ("http.followRedirects", "false")]
    header = None
    try:
        if token:
            header = "Authorization: Basic " + base64.b64encode(("x-access-token:" + token).encode()).decode()
            settings.extend(("http." + scope + ".extraHeader", header) for scope in dict.fromkeys(scopes))
        git_env["GIT_CONFIG_COUNT"] = str(len(settings))
        for i, (key, value) in enumerate(settings):
            git_env[f"GIT_CONFIG_KEY_{i}"] = key
            git_env[f"GIT_CONFIG_VALUE_{i}"] = value
        return runner([sys.executable, "-c", GITHUB_GIT_WORKER, *command], logs,
                      name=name, env=git_env, cwd=cwd, capture_json=capture_json)
    except Exception:
        raise RuntimeError("GitHub access failed. Check the logged URL/revision error and Colab Secrets "
                           "GITHUB_TOKEN access, selected repositories, Contents: Read-only, and expiry.") from None
    finally:
        token = header = None
        settings.clear()
        git_env.clear()


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None, bootstrap=False):
    release = globals().get('RELEASE_RECORD')
    if globals().get('RELEASE_REQUIRED') and release is None and not bootstrap:
        raise RuntimeError('Run the Colab revision check before starting the experiment.')

    def runner(command, logs, **options):
        metadata = None if release is None else {'colab_release': release}
        return run_process(command, logs, extra_record=metadata, **options)

    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    if command and command[0] == 'git':
        repositories = [globals().get('REPO_URL', 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'),
                        *globals().get('GITHUB_AUTH_REPOS', [])]
        return run_github_git(command, LOGS, runner=runner, repositories=repositories,
                              name=name, capture_json=capture_json, env=env, cwd=cwd)
    return runner(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/training/bootstrap')

In [ ]:
#@title Mount Google Drive

from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title Experiment and training stages {display-mode:"form"}

#@markdown ### Experiment
EXPERIMENT_ID = 'experiment-001' #@param {type:"string"}
TRANSFER_EXPERIMENT_ID = '' #@param {type:"string"}
CALIBRATION_EXPERIMENT_ID = '' #@param {type:"string"}

#@markdown ### Data and audio preparation
PREPARE_S2UT = True #@param {type:"boolean"}
PREPARE_AUDIO_PACKS = True #@param {type:"boolean"}
AUDIO_PACK_WORKERS = 2 #@param {type:"integer"}
DATA_CONFIG = 'configs/data/colab.yaml' #@param {type:"string"}
CORPUS_ROOT_OVERRIDE = '' #@param {type:"string"}
SOURCE_MANIFEST_OVERRIDE = '' #@param {type:"string"}
RELEASE_SUMMARY_OVERRIDE = '' #@param {type:"string"}

#@markdown ### T2TT teacher model
RUN_T2TT = False #@param {type:"boolean"}
T2TT_RUN_ID = 't2tt-001' #@param {type:"string"}
T2TT_MAX_UPDATES = 200000 #@param {type:"integer"}
T2TT_TOKENIZER = 'sentencepiece' #@param ['character', 'sentencepiece']
T2TT_SOURCE_MANIFEST_OVERRIDE = '' #@param {type:"string"}

#@markdown ### S2UT Baseline
RUN_BASELINE = False #@param {type:"boolean"}
BASELINE_NOTEBOOK_REVISION = '' #@param {type:"string"}
BASELINE_EXECUTION_LOGS = '' #@param {type:"string"}

#@markdown ### Teacher artifacts
RUN_TEACHER_ARTIFACTS = False #@param {type:"boolean"}

#@markdown ### Top-p diagnostics and calibration
RUN_MASK_TOPP_DIAGNOSTICS = False #@param {type:"boolean"}
#@markdown Select the value used by the saved calibration, or choose one after reviewing a new diagnostic report.
MASK_TOP_P = None #@param ["None", "0.8", "0.9", "0.95"] {type:"raw"}
RUN_ALIGNMENT_CALIBRATION = False #@param {type:"boolean"}
CALIBRATION_BATCHES = 16 #@param {type:"integer"}

#@markdown ### Transfer training conditions
TRANSFER_KL_LAMBDA = 1.0 #@param {type:"number"}
TRANSFER_MASK_LAMBDA = 10.0 #@param {type:"number"}
RUN_DISTILLATION = False #@param {type:"boolean"}
RUN_ANNEALED_DISTILLATION = False #@param {type:"boolean"}
RUN_MASK_DISTILLATION = False #@param {type:"boolean"}

#@markdown ### Pilot and checkpoint recovery
TRANSFER_RUN_MODE = 'pilot' #@param ['pilot', 'full']
TRANSFER_PILOT_UPDATES = 20000 #@param {type:"integer"}
TRANSFER_DIAGNOSTIC_STEPS = [1000, 5000, 10000, 20000]
TRANSFER_RECOVERY_CHECK = 'off' #@param ['off', 'save', 'resume']

#@markdown ### Evaluation and diagnostics
TRANSFER_ONLINE_BLEU = True #@param {type:"boolean"}
EVALUATE_T2TT = False #@param {type:"boolean"}
EVALUATE_TRANSFER_PILOT_DEV = False #@param {type:"boolean"}
TRANSFER_PILOT_ASR_BLEU = False #@param {type:"boolean"}
RUN_ALIGNMENT_DIAGNOSTICS = False #@param {type:"boolean"}

#@markdown ### Weights & Biases (optional)
ENABLE_WANDB = False #@param {type:"boolean"}
WANDB_PROJECT = 'alignment-transfer-s2ut' #@param {type:"string"}
WANDB_ENTITY = '' #@param {type:"string"}

# Validate settings before setup.
if TRANSFER_EXPERIMENT_ID:
    import re
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', TRANSFER_EXPERIMENT_ID):
        raise ValueError('TRANSFER_EXPERIMENT_ID must use letters, digits, underscores, or hyphens')
if CALIBRATION_EXPERIMENT_ID:
    import re
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', CALIBRATION_EXPERIMENT_ID):
        raise ValueError('CALIBRATION_EXPERIMENT_ID must use letters, digits, underscores, or hyphens')
if TRANSFER_RECOVERY_CHECK not in ('off', 'save', 'resume'):
    raise ValueError('TRANSFER_RECOVERY_CHECK must be off, save, or resume')
if TRANSFER_RECOVERY_CHECK != 'off' and (
        PREPARE_AUDIO_PACKS or RUN_BASELINE or TRANSFER_RUN_MODE != 'pilot'
        or sum((RUN_DISTILLATION, RUN_ANNEALED_DISTILLATION, RUN_MASK_DISTILLATION)) != 1):
    raise ValueError('Recovery check requires GPU Pilot mode, RUN_BASELINE=False, and exactly one transfer condition')
if TRANSFER_RUN_MODE not in ('pilot', 'full'):
    raise ValueError('TRANSFER_RUN_MODE must be pilot or full')
if TRANSFER_PILOT_ASR_BLEU and not EVALUATE_TRANSFER_PILOT_DEV:
    raise ValueError('Enable EVALUATE_TRANSFER_PILOT_DEV before requesting ASR-BLEU')
NEED_TEACHER = RUN_TEACHER_ARTIFACTS or RUN_MASK_TOPP_DIAGNOSTICS or RUN_ALIGNMENT_CALIBRATION or RUN_DISTILLATION or RUN_ANNEALED_DISTILLATION or RUN_MASK_DISTILLATION or RUN_ALIGNMENT_DIAGNOSTICS
NEED_TEXT = RUN_T2TT or EVALUATE_T2TT or NEED_TEACHER
if NEED_TEACHER and T2TT_TOKENIZER != 'sentencepiece':
    raise ValueError('Select SentencePiece T2TT for the teacher used in the main comparison.')
if not PREPARE_S2UT and (RUN_BASELINE or NEED_TEACHER):
    raise ValueError('Set PREPARE_S2UT=True for Baseline or transfer training.')

if isinstance(AUDIO_PACK_WORKERS, bool) or not isinstance(AUDIO_PACK_WORKERS, int) or not 1 <= AUDIO_PACK_WORKERS <= 4:
    raise ValueError("Set AUDIO_PACK_WORKERS to an integer from 1 to 4.")


In [ ]:
#@title Storage and pinned environment
import os
import sys
for value in (EXPERIMENT_ID, globals().get('T2TT_RUN_ID', 'text')):
    if not value or Path(value).name != value or value in ('.', '..'):
        raise ValueError('Use a single directory name for the experiment ID.')
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
NOTEBOOK_UPDATED_AT = '2026-11-24T14:21+09:00'
REPO_REVISION = 'b9a6b6075655cfa807c8b696cb1a188c83f3702c'
REPO = Path('/content/alignment-transfer-s2ut')
PYTHON = '/content/alignment-venv/bin/python'
ROOT = str(Path('/content/alignment-transfer-work') / EXPERIMENT_ID)
PERSISTENT = str(Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / EXPERIMENT_ID)
LOGS = Path(ROOT) / 'execution'
BENCHMARK_RUNTIME_URL = 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '0c0024ea888206149a3119bf096b86ba66d63b66'
DATA = Path(ROOT) / 'data'
COMMON = DATA / 'common'
BENCHMARK_UNITS = DATA / 's2ut/units'
PREPARED = DATA / 's2ut/fairseq'
BACKUP = Path(PERSISTENT) / 'baseline'
RESTORED = Path(ROOT) / 'baseline'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
TRANSFER_SUFFIX = '-run-' + TRANSFER_EXPERIMENT_ID if TRANSFER_EXPERIMENT_ID else ''
SCOPE_SUFFIX = '-spm-all-layers' + TRANSFER_SUFFIX
RESEARCH_ROOT = Path(ROOT) / ('research-v3' + SCOPE_SUFFIX)
calibration_id = CALIBRATION_EXPERIMENT_ID or TRANSFER_EXPERIMENT_ID
CALIBRATION_SCOPE_SUFFIX = '-spm-all-layers' + ('-run-' + calibration_id if calibration_id else '')
CALIBRATION_RESEARCH_ROOT = Path(ROOT) / ('research-v3' + CALIBRATION_SCOPE_SUFFIX)
REUSE_CALIBRATION = CALIBRATION_SCOPE_SUFFIX != SCOPE_SUFFIX
TOPP_ROOT = CALIBRATION_RESEARCH_ROOT / 'pilot-top-p-masked'
TOPP_REPORT = TOPP_ROOT / 'top-p.json'
PREVIOUS_CALIBRATION = CALIBRATION_RESEARCH_ROOT / 'pilot-calibration-masked' / 'alignment-calibration.json'
CALIBRATION_ROOT = CALIBRATION_RESEARCH_ROOT / 'pilot-calibration-fp16'
CALIBRATION = CALIBRATION_ROOT / 'alignment-calibration.json'
METHOD_LABELS = [
    ('distillation', 'KL-based Attention Distillation'),
    ('annealed_distillation', 'Annealed KL-based Attention Distillation'),
    ('mask_distillation', 'Attention-level Knowledge Distillation')]
os.environ.update(HF_HOME='/content/huggingface', MPLBACKEND='Agg', PYTHONUNBUFFERED='1', PYTHONUTF8='1',
    MPLCONFIGDIR='/content/matplotlib-cache', S2ST_DRIVE_SAFE='1',
    S2ST_BACKUP_MIN_SECONDS='600', S2ST_BACKUP_MAX_GB='3072')
RELEASE_REQUIRED = True
RELEASE_RECORD = None


## 1. Environment

Install Python, fairseq, and the benchmark runtime. GPU preflight checks a small update, checkpoint save, and resume.

In [ ]:
#@title Verify Git revisions and record the notebook update time
"""Validate the two Colab checkouts and synchronize the embedded bootstrap."""

from datetime import datetime, timezone
import json
from pathlib import Path
import re
import sys
import uuid


GIT_CHECKOUT_STATE = r"""
import json
import subprocess
import sys

def read_git(path, *arguments):
    result = subprocess.run(
        ["git", "-C", path, *arguments], stdout=subprocess.PIPE, check=False,
        encoding="utf-8", errors="replace", text=True)
    return result.stdout if result.returncode == 0 else None

states = []
for path in sys.argv[1:]:
    head = read_git(path, "rev-parse", "--verify", "HEAD")
    status = read_git(path, "status", "--porcelain=v1", "-z", "--untracked-files=all", "--ignore-submodules=none")
    states.append({
        "actual_head": head.strip() if head is not None else None,
        "worktree": ("unknown" if status is None else "dirty" if status else "clean"),
        "status_entries": status.rstrip("\0").split("\0") if status else [],
    })
print(json.dumps(states, ensure_ascii=True))
"""


def prepare_colab_release(execute, log_event, *, notebook, notebook_updated_at, experiment_id, transfer_id,
                          alignment, benchmark, logs, persistent):
    """Checkout only the pinned SHAs, record both observations, and fail closed."""
    checkouts = dict(alignment=alignment, benchmark_runtime=benchmark)
    errors = []
    for key, (path, url, revision) in checkouts.items():
        if not re.fullmatch(r"[0-9a-f]{40}", revision):
            errors.append(f"{key}: expected a full lowercase commit SHA")
    if not errors:
        for key, (path, url, revision) in checkouts.items():
            label = key.replace("_", "-")
            try:
                if not (path / ".git").exists():
                    execute("git", "clone", "--no-checkout", url, path, name="clone-" + label)
                execute("git", "-C", path, "fetch", "origin", revision, name="fetch-" + label)
                execute("git", "-C", path, "checkout", "--detach", revision, name="checkout-" + label)
            except Exception:
                errors.append(f"{key}: Git checkout failed; see the command log")
    try:
        result = execute(sys.executable, "-c", GIT_CHECKOUT_STATE,
                         *(value[0] for value in checkouts.values()),
                         name="inspect-colab-revisions", capture_json=True)
        states = json.loads(result.stdout)
    except Exception:
        errors.append("could not inspect the repositories; see the command log")
        states = [{"actual_head": None, "worktree": "unknown", "status_entries": []} for _ in checkouts]
    repositories = {}
    for (key, (_, _, pin)), state in zip(checkouts.items(), states, strict=True):
        repositories[key] = {"pinned_revision": pin, **state}
        if state["actual_head"] != pin:
            errors.append(f"{key}: HEAD differs from the pinned revision")
        if state["worktree"] != "clean":
            errors.append(f"{key}: worktree is {state['worktree']}")
    report = dict(schema="colab-release-v2", run_id=uuid.uuid4().hex,
                  started_utc=datetime.now(timezone.utc).isoformat(), notebook=notebook,
                  notebook_updated_at=notebook_updated_at, experiment_id=experiment_id,
                  transfer_experiment_id=transfer_id, status="FAIL" if errors else "PASS",
                  repositories=repositories, errors=errors)
    payload = json.dumps(report, ensure_ascii=True, indent=2, allow_nan=False) + "\n"
    filename = "colab-release-" + report["run_id"] + ".json"
    for root in (Path(logs), Path(persistent) / "execution-provenance"):
        root.mkdir(parents=True, exist_ok=True)
        with (root / filename).open("x", encoding="utf-8") as target:
            target.write(payload)
    log_event("colab-revision-check", report["status"], logs, json.dumps(report, ensure_ascii=True))
    print(payload, end="", flush=True)
    if errors:
        raise RuntimeError("Colab revision check failed. Resolve the reported Git state before running the experiment.")
    return report

# Start the revision check
RELEASE_RECORD = None
RELEASE_RECORD = prepare_colab_release(
    lambda *args, **kwargs: execute(*args, bootstrap=True, **kwargs), log_event,
    notebook='training', notebook_updated_at=NOTEBOOK_UPDATED_AT,
    experiment_id=EXPERIMENT_ID, transfer_id=TRANSFER_EXPERIMENT_ID,
    alignment=(REPO, REPO_URL, REPO_REVISION),
    benchmark=(BENCHMARK_RUNTIME, BENCHMARK_RUNTIME_URL, BENCHMARK_RUNTIME_REVISION),
    logs=LOGS, persistent=PERSISTENT)


In [ ]:
#@title Set up Python and fairseq
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title Install the pinned benchmark runtime
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
#@title Check the saved Baseline

if not PREPARE_AUDIO_PACKS and NEED_TEACHER and not RUN_BASELINE:
    print(execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.transfer_pilot', 'baseline-ready',
                       '--backup', str(BACKUP), name='baseline-completion-check'))
else:
    log_event('baseline-completion-check', 'skipped', LOGS, 'transfer stages are disabled or Baseline training was requested')

In [ ]:
#@title Load corpus settings

config = execute_json(PYTHON, '-c',
    "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1], encoding='utf-8'))))",
    DATA_CONFIG, name='data-settings')
needs_speech = globals().get('PREPARE_S2UT', globals().get('EVALUATE_S2UT', False))
root_value = CORPUS_ROOT_OVERRIDE or config.get('corpus_root')
manifest_value = SOURCE_MANIFEST_OVERRIDE or config.get('accepted_manifest')
summary_value = RELEASE_SUMMARY_OVERRIDE or config.get('release_summary')
if needs_speech and (not root_value or not manifest_value):
    raise ValueError('Set the corpus root and manifest in DATA_CONFIG, or set both CORPUS_ROOT_OVERRIDE and SOURCE_MANIFEST_OVERRIDE.')
CORPUS_ROOT = Path(root_value) if root_value else None
SOURCE_MANIFEST = Path(manifest_value) if manifest_value else None
RELEASE_SUMMARY = Path(summary_value) if summary_value else None
if CORPUS_ROOT and any(path.resolve().is_relative_to(CORPUS_ROOT.resolve()) for path in (Path(ROOT), Path(PERSISTENT))):
    raise ValueError('Choose an output directory outside the source corpus.')
os.environ.pop('S2ST_INPUT_MAP', None)
execute(PYTHON, '-m', 'pip', 'install', 'editdistance==0.8.1', 'librosa==0.10.2.post1',
    'accelerate==1.12.0', 'pandas==2.2.3', 'psutil==7.2.2', name='runtime-dependencies')
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
#@title Check runtime dependencies and native training

execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark',
    *([] if PREPARE_AUDIO_PACKS else ['--cuda']), name='corpus-environment')
if PREPARE_AUDIO_PACKS:
    log_event('corpus-native-fp16-test', 'skipped', LOGS, 'CPU audio preparation')
else:
    execute(PYTHON, '-c',
        'from direct_s2st.s2ut.paper_recipe import preflight_dependencies; print(preflight_dependencies())',
        name='corpus-paper-dependencies')
    execute(PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
        'tests/test_corpus_runtime.py::test_corpus_release_to_native_recipe',
        '-v', '-s', '-m', 'gpu', '--tb=short', '--basetemp', '/content/corpus-native-fp16-test',
        cwd=REPO, name='corpus-native-fp16-test')


In [ ]:
#@title Run, save, and restore helpers
def corpus_stage(name, module, *arguments, work, restore_only=False):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(work),
        '--backup', str(Path(PERSISTENT) / 'stages' / name)]
    if restore_only:
        command.append('--restore-only')
    execute(*command, '--', PYTHON, '-m', module, *map(str, arguments), name=name)

def corpus_common(restore_only=False):
    os.environ['CORPUS_ROOT'] = str(CORPUS_ROOT)
    summary_args = ['--summary', str(RELEASE_SUMMARY)] if RELEASE_SUMMARY else []
    corpus_stage('common', 'alignment_transfer_s2ut.benchmark_data', 'corpus',
        '--source', SOURCE_MANIFEST, '--corpus-root', CORPUS_ROOT, *summary_args,
        '--output', COMMON, '--resume', work=COMMON, restore_only=restore_only)
    lock = json.loads((COMMON / 'dataset-lock.json').read_text(encoding='utf-8'))
    expected = config.get('expected_pairs')
    if expected is not None and lock['total_pairs'] != expected:
        raise ValueError(f"train/dev count {lock['total_pairs']} differs from expected_pairs={expected}")
    print({'pairs': lock['total_pairs'], 'splits': lock['splits'],
        'estimated_waveform_GiB': round(lock['wav_bytes_estimate'] / 1024**3, 1)})

def corpus_stage_audio(create=False):
    staged = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-corpus',
        '--common', str(COMMON), '--corpus-root', str(CORPUS_ROOT), '--packs', str(Path(PERSISTENT) / 'wav-packs'),
        '--local', str(Path(ROOT) / 'staged-inputs'), *(['--create-audio', '--audio-pack-workers', str(AUDIO_PACK_WORKERS)] if create else ['--stream-audio']), name='audio-staging')
    if not create:
        os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

def corpus_prepare(restore_only=False):
    corpus_stage('unit-extraction', 'alignment_transfer_s2ut.corpus_runtime', 'units',
        '--common', COMMON, '--output', BENCHMARK_UNITS, '--cache', '/content/alignment-assets',
        work=BENCHMARK_UNITS, restore_only=restore_only)
    corpus_stage('fairseq-data', 'alignment_transfer_s2ut.corpus_runtime', 'prepare',
        '--common', COMMON, '--units', BENCHMARK_UNITS, '--output', PREPARED,
        work=PREPARED, restore_only=restore_only)

def corpus_baseline(restore_only=False):
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.corpus_runtime', 'baseline',
        '--data', str(DATA), '--work', str(RESTORED), '--backup', str(BACKUP),
        *(['--restore-only'] if restore_only else ['--train']), name='baseline')

def prepare_wandb():
    from google.colab import userdata
    try:
        key = userdata.get('WANDB_API_KEY')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        key = None
    if not key or not WANDB_PROJECT.strip():
        log_event('wandb', 'skipped', LOGS, 'Set the WANDB_API_KEY Colab secret and a project name')
        return None
    directory = Path('/content/alignment-wandb-venv')
    python = directory / 'bin/python'
    import subprocess
    try:
        execute('uv', 'venv', '--python', PYTHON, '--allow-existing', directory, name='wandb-environment')
        execute('uv', 'pip', 'install', '--python', python, 'wandb==0.31.0', 'filelock==4.1.0', name='wandb-install')
    except (OSError, subprocess.CalledProcessError):
        log_event('wandb', 'failed', LOGS, 'SDK setup failed. See the setup log. Training metrics stay local.')
        return None
    return python, key

def run(name, module, *arguments, gpu=False, env=None, restore_only=False, wandb_tracking=None, after_training=None):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if restore_only:
                wrapper.append('--restore-only')
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
            if wandb_tracking is not None:
                python, key, contract, method = wandb_tracking
                group = f'{EXPERIMENT_ID}/{TRANSFER_EXPERIMENT_ID or "default"}'
                env = dict(os.environ if env is None else env, WANDB_API_KEY=key)
                bleu_results = Path(PERSISTENT) / 'transfer-bleu' / (TRANSFER_EXPERIMENT_ID or 'default')
                post = ['--after-training', json.dumps(list(map(str, after_training)))] if after_training else []
                payload = [python, REPO / 'tools/wandb_transfer.py', '--work', work,
                    '--backup', Path(PERSISTENT) / 'stages' / name, '--logs', LOGS,
                    '--group', group, '--method', method, '--contract', contract,
                    '--project', WANDB_PROJECT.strip(), '--entity', WANDB_ENTITY.strip(),
                    '--bleu-results', bleu_results, *post, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return execute(*command, '--', *payload, name='notebook:' + name, env=env)

## 2. T2TT

Select the saved SentencePiece T2TT run to restore it. For a new model, set a new `T2TT_RUN_ID` and `RUN_T2TT=True`. Use `EVALUATE_T2TT=True` for dev BLEU, translation examples, and learning curves.

For T2TT only, set `PREPARE_S2UT=False` and `PREPARE_AUDIO_PACKS=False`.

In [ ]:
#@title Resolve T2TT inputs

T2TT_SOURCE_MANIFEST = T2TT_SOURCE_MANIFEST_OVERRIDE or config.get('teacher_source_manifest', '')
TEXT_ROOT = Path('/content/alignment-transfer-work') / T2TT_RUN_ID
TEXT_PERSISTENT = Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / T2TT_RUN_ID
TEXT_LOGS = TEXT_ROOT / 'execution'
TEXT_MANIFEST = TEXT_ROOT / 'text-manifest/manifest.jsonl'
TEXT_CONFIG = REPO / 'configs/t2tt/fisher.yaml'
TEXT_TRAIN_STAGE = 't2tt-' + T2TT_TOKENIZER
TEXT_WORK = TEXT_ROOT / TEXT_TRAIN_STAGE
TEXT_BACKUP = TEXT_PERSISTENT / 'stages' / TEXT_TRAIN_STAGE
TEXT_CHECKPOINT = TEXT_WORK / 'checkpoint_best.pt'
TEXT_EVAL_ROOT = TEXT_ROOT / ('dev-' + T2TT_TOKENIZER)
TEXT_EVAL_BACKUP = TEXT_PERSISTENT / 'stages' / ('dev-' + T2TT_TOKENIZER)
TEXT_DEV_METRICS = TEXT_EVAL_ROOT / 'dev-text.json'
T2TT_SNAPSHOT_UPDATES = 10000
T2TT_KEEP_LAST = 2
if NEED_TEXT and not T2TT_SOURCE_MANIFEST:
    raise ValueError('Set T2TT_SOURCE_MANIFEST_OVERRIDE or teacher_source_manifest in DATA_CONFIG.')


In [ ]:
#@title Prepare the T2TT text manifest

if not PREPARE_AUDIO_PACKS and NEED_TEXT:
    prepared = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest', '--input', T2TT_SOURCE_MANIFEST, '--output', str(TEXT_MANIFEST), '--resume', name='prepare-external-text')
    print(json.dumps(prepared, ensure_ascii=False, indent=2))
else:
    log_event('text-input', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title Train T2TT or find its saved run

if not PREPARE_AUDIO_PACKS and NEED_TEXT:
    if RUN_T2TT:
        training_args = ['--manifest', str(TEXT_MANIFEST), '--config', str(TEXT_CONFIG), '--output', str(TEXT_WORK),
            '--backend', 'fairseq', '--device', 'cuda', '--tokenizer', T2TT_TOKENIZER, '--max-updates', str(T2TT_MAX_UPDATES),
            '--batch-size', '8', '--update-frequency', '4', '--max-tokens', '20000', '--loader-workers', '2',
            '--validation-interval', '1000', '--save-interval-updates', '100', '--resume-if-exists']
        if T2TT_TOKENIZER == 'sentencepiece':
            training_args += ['--source-vocab-size', '8000', '--target-vocab-size', '8000', '--sentencepiece-model-type', 'unigram']
        execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', TEXT_TRAIN_STAGE, '--logs', str(TEXT_LOGS),
            '--monitor-gpu', '--', PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'train',
            '--work', str(TEXT_WORK), '--backup', str(TEXT_BACKUP), '--snapshot-updates', str(T2TT_SNAPSHOT_UPDATES), '--keep-last', str(T2TT_KEEP_LAST),
            '--', PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', *training_args, name='text-t2tt-training')
    else:
        log_event(TEXT_TRAIN_STAGE, 'skipped', TEXT_LOGS, 'training disabled; restore completed checkpoint below')
else:
    log_event('text-training', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title Restore the verified T2TT checkpoint

if not PREPARE_AUDIO_PACKS and NEED_TEXT:
    restored = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
        '--work', str(TEXT_WORK), '--backup', str(TEXT_BACKUP), '--expected-updates', str(T2TT_MAX_UPDATES), '--dev-backup', str(TEXT_EVAL_BACKUP), '--dev-output', str(TEXT_DEV_METRICS), name='restore-text-t2tt')
    status = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'status', '--work', str(TEXT_WORK),
        '--target-updates', str(T2TT_MAX_UPDATES), name='verify-text-t2tt-complete')
    print(status)
else:
    log_event('text-restore', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title Evaluate T2TT on dev

if not PREPARE_AUDIO_PACKS and EVALUATE_T2TT:
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(TEXT_EVAL_ROOT), '--backup', str(TEXT_EVAL_BACKUP), '--',
        PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', str(TEXT_MANIFEST), '--checkpoint', str(TEXT_CHECKPOINT),
        '--split', 'dev', '--beam-size', '5', '--length-penalty', '1', '--device', 'cuda', '--output', str(TEXT_DEV_METRICS), '--resume', name='text-dev-bleu')
    metrics = json.loads(TEXT_DEV_METRICS.read_text(encoding='utf-8'))
    print(json.dumps({k:v for k,v in metrics.items() if k not in ('evaluation_contract',)}, ensure_ascii=False, indent=2))
else:
    log_event('text-evaluation', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title View translation examples and learning curves

if not PREPARE_AUDIO_PACKS and EVALUATE_T2TT:
    import pandas as pd
    from IPython.display import display, Image
    predictions = [json.loads(line) for line in TEXT_DEV_METRICS.with_name(TEXT_DEV_METRICS.stem + '-predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    references = {row['pair_id']:row['en_text'] for row in (json.loads(line) for line in TEXT_MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()) if row['split']=='dev'}
    display(pd.DataFrame([{'pair_id':r['pair_id'], 'JA':r['source_text'], 'Reference EN':references[r['pair_id']],
        'T2TT EN':r['hypothesis'], 'EOS':r['eos_reached']} for r in predictions[:10]]))
    diagnostic = TEXT_WORK / 'tokenization.json'
    if diagnostic.is_file():
        tokenization = json.loads(diagnostic.read_text(encoding='utf-8'))
        print('actual vocab sizes:', tokenization['source']['actual_vocab_size'], tokenization['target']['actual_vocab_size'])
        display(pd.DataFrame(tokenization['statistics']).T)
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'report', '--work', str(TEXT_WORK),
        '--output', str(TEXT_WORK / 'diagnostics'), name='text-learning-curve')
    display(Image(filename=str(TEXT_WORK / 'diagnostics/learning-curve.png')))
else:
    log_event('text-examples', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 3. Speech and training data

Prepare QC-accepted train/dev data, WAV ZIPs, HuBERT L6/KM100 units, dictionaries, and fairseq data. On GPU, index audio ZIPs and fetch and verify them on demand. The initial run prepares units and dictionaries. On resume, restore saved preprocessing and checkpoints while the loader fetches upcoming batches.

In [ ]:
#@title Prepare or index WAV ZIPs

if PREPARE_S2UT:
    corpus_common()
    corpus_stage_audio(create=PREPARE_AUDIO_PACKS)
    if PREPARE_AUDIO_PACKS:
        print('WAV ZIPs are ready. Switch to a GPU runtime, set PREPARE_AUDIO_PACKS=False, and rerun from the top.')
else:
    log_event('audio-staging', 'skipped', LOGS, 'PREPARE_S2UT=False')


In [ ]:
#@title Prepare units and fairseq data

if PREPARE_S2UT:
    if PREPARE_AUDIO_PACKS:
        log_event('corpus-s2ut-preparation', 'skipped', LOGS, 'CPU audio preparation')
    else:
        corpus_prepare()
        execute(PYTHON, '-c',
            'import sys; from alignment_transfer_s2ut.benchmark import prepared_identity; '
            'print(prepared_identity(sys.argv[1])["data_lock"]["splits"])',
            str(PREPARED), name='corpus-registration-preflight')
else:
    log_event('speech-preparation', 'skipped', LOGS, 'PREPARE_S2UT=False')


## 4. Baseline

Set `RUN_BASELINE=True` to train or resume the 400,000-update Baseline. Restore and register the completed Baseline before preparing the teacher or training transfer conditions.

In [ ]:
#@title Train or restore the Baseline

if PREPARE_AUDIO_PACKS:
    log_event('baseline', 'skipped', LOGS, 'CPU audio preparation')
elif RUN_BASELINE or NEED_TEACHER:
    corpus_baseline(restore_only=not RUN_BASELINE)
else:
    log_event('baseline', 'skipped', LOGS, 'training and teacher stages disabled')

In [ ]:
#@title Register the Baseline checkpoint

if not PREPARE_AUDIO_PACKS and (RUN_BASELINE or NEED_TEACHER):
    run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
        '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
        '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
        '--output', BASELINE, '--resume')
    baseline = json.loads(BASELINE.read_text(encoding='utf-8'))
    print({key:baseline[key] for key in ('checkpoint_updates','training_budget','selection_policy')})
    if NEED_TEACHER:
        if BASELINE_NOTEBOOK_REVISION:
            import re
            if not re.fullmatch(r'[0-9a-f]{40}', BASELINE_NOTEBOOK_REVISION):
                raise ValueError('BASELINE_NOTEBOOK_REVISION must be the full original alignment commit SHA')
            execute('git', '-C', str(REPO), 'fetch', 'origin', BASELINE_NOTEBOOK_REVISION,
                    name='fetch-baseline-provenance-revision')
        report = execute_json(PYTHON, str(REPO / 'tools/review_transfer_pilot.py'), 'baseline',
            '--baseline', str(BASELINE), '--backup', str(BACKUP), '--repo', str(REPO),
            '--output', str(RESEARCH_ROOT / 'baseline-provenance'),
            *(['--revision', BASELINE_NOTEBOOK_REVISION] if BASELINE_NOTEBOOK_REVISION else []),
            *(['--logs', BASELINE_EXECUTION_LOGS] if BASELINE_EXECUTION_LOGS else []), name='baseline-provenance')
        print(report)
    BACKBONE_CONFIG.write_text(json.dumps({'backbone':baseline['backbone']}),encoding='utf-8')
else:
    log_event('baseline-registration', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title Prepare the speech manifest and unit artifacts

if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
        '--common', COMMON, '--prepared', PREPARED, '--output', MANIFEST, '--resume')
    run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
        '--benchmark-units', BENCHMARK_UNITS, '--prepared', PREPARED,
        '--output', Path(ROOT) / 'units', '--resume')
else:
    log_event('speech-artifacts', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 5. Teacher

Verify the original T2TT train and dev hashes and matching speech IDs, splits, and text. Extract all-layer attention, align CTC, project attention onto speech time, and run quality diagnostics.

In [ ]:
#@title Set the teacher artifact paths

T2TT_ACTIVE_ROOT = Path(ROOT) / 'teacher-text-model'
T2TT_CHECKPOINT = T2TT_ACTIVE_ROOT / 'checkpoint_best.pt'
T2TT_ATTENTION_STAGE = 't2tt-attention-all-layers'
T2TT_ATTENTION_ROOT = Path(ROOT) / T2TT_ATTENTION_STAGE
CTC_STAGE = 'speech-text-alignment'
CTC_ROOT = Path(ROOT) / CTC_STAGE
TEACHER_STAGE = 'teacher-all-layers'
TEACHER_ROOT = Path(ROOT) / TEACHER_STAGE
TEACHER_AUDIT_STAGE = 'teacher-projection-audit-v2'
TEACHER_AUDIT_ROOT = Path(ROOT) / TEACHER_AUDIT_STAGE
LAYER_FLAGS = ['--all-layers']
RESEARCH_CONDITIONS = [(method,label,globals()['RUN_' + method.upper()]) for method,label in METHOD_LABELS]
NEED_T2TT = NEED_TEACHER

In [ ]:
#@title Extract T2TT attention and align speech with CTC

if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    teacher_stages = (T2TT_ATTENTION_STAGE, CTC_STAGE, TEACHER_STAGE)
    saved_teacher = all(any((Path(PERSISTENT) / 'stages' / stage).glob('*/snapshot.json')) for stage in teacher_stages)
    if not saved_teacher:
        if not RUN_TEACHER_ARTIFACTS:
            raise ValueError('Review the T2TT evaluation and set RUN_TEACHER_ARTIFACTS=True to build the teacher for the first time.')
        if not TEXT_DEV_METRICS.is_file() or json.loads(TEXT_DEV_METRICS.read_text(encoding='utf-8')).get('checkpoint_hash') != status['checkpoint_hash']:
            raise ValueError('Set EVALUATE_T2TT=True and complete dev evaluation for the selected T2TT run.')
    text_manifest = Path(ROOT) / 'teacher-text/manifest.jsonl'
    execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest', '--input', str(TEXT_MANIFEST),
        '--output', str(text_manifest), '--resume', name='teacher-text')
    snapshots = sorted(TEXT_BACKUP.glob('*/snapshot.json'), reverse=True)
    if not snapshots:
        raise FileNotFoundError('No verified T2TT snapshot was found.')
    execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.corpus_runtime', 'restore-teacher',
        '--snapshot', str(snapshots[0].parent), '--output', str(T2TT_ACTIVE_ROOT), '--updates', str(T2TT_MAX_UPDATES),
        '--source-manifest', str(text_manifest), '--speech-manifest', str(MANIFEST),
        '--backbone', str(BACKBONE_CONFIG), name='verify-teacher')
    run(T2TT_ATTENTION_STAGE, 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
        '--checkpoint', T2TT_CHECKPOINT, '--output', T2TT_ATTENTION_ROOT, '--device', 'cuda',
        '--resume', *LAYER_FLAGS, '--teacher-training-manifest', text_manifest, gpu=True)
    run(CTC_STAGE, 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST,
        '--config', 'configs/alignment/ctc.yaml', '--output', CTC_ROOT, '--device', 'cuda', '--resume', gpu=True)
else:
    log_event('teacher-extraction', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title Build and verify the teacher artifacts

if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    if NEED_T2TT:
        run(TEACHER_STAGE, 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
            '--mt-attention', T2TT_ATTENTION_ROOT, '--alignment', CTC_ROOT, '--units', Path(ROOT) / 'units',
            '--config', BACKBONE_CONFIG, '--output', TEACHER_ROOT, '--resume')
        run(TEACHER_AUDIT_STAGE, 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
            '--teacher', TEACHER_ROOT, '--units', Path(ROOT) / 'units', '--config', BACKBONE_CONFIG,
            '--split', 'dev', '--seed', 1, '--max-samples', 500, '--heatmaps', 20,
            '--prior-smoothing', 0, '--output', TEACHER_AUDIT_ROOT, '--overwrite', *LAYER_FLAGS)
    else:
        log_event('teacher-preparation', 'skipped', LOGS, 'teacher stage not requested')
else:
    log_event('teacher-projection', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 6. Top-p diagnostics and calibration

- **Reuse saved results.** Set `CALIBRATION_EXPERIMENT_ID` to the original run ID and leave both generation switches off. Select the saved calibration’s `MASK_TOP_P` in **Experiment and training stages**. For the fixed-coefficient run documented in [Explicit transfer coefficients](../../docs/FIXED_KL.md), choose **0.8**.
- **Run a new calibration.** Enable `RUN_MASK_TOPP_DIAGNOSTICS` and inspect the report. Select `MASK_TOP_P`, enable `RUN_ALIGNMENT_CALIBRATION`, and rerun the settings and calibration cells. The notebook checks FP16 loss scales against FP32 before saving λ and τ.

In [ ]:
#@title Run or restore Top-p diagnostics and calibration

if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    CALIBRATION_INPUTS = ['--baseline', str(BASELINE), '--manifest', str(MANIFEST),
        '--units', str(Path(ROOT) / 'units'), '--teacher', str(TEACHER_ROOT), '--batches', str(CALIBRATION_BATCHES)]
    topp_stage = 'research-v5-top-p' + CALIBRATION_SCOPE_SUFFIX
    topp_saved = any((Path(PERSISTENT) / 'stages' / topp_stage).glob('*/snapshot.json'))
    calibration_stage = 'research-v6-calibration-fp16' + CALIBRATION_SCOPE_SUFFIX
    previous_stage = 'research-v5-calibration' + CALIBRATION_SCOPE_SUFFIX
    calibration_saved = any((Path(PERSISTENT) / 'stages' / calibration_stage).glob('*/snapshot.json'))
    if REUSE_CALIBRATION and (RUN_MASK_TOPP_DIAGNOSTICS or RUN_ALIGNMENT_CALIBRATION):
        raise ValueError('Reusing a separate calibration requires both diagnostic and calibration switches off')
    if RUN_MASK_TOPP_DIAGNOSTICS or topp_saved or REUSE_CALIBRATION:
        run(topp_stage, 'alignment_transfer_s2ut.alignment_calibration', 'top-p',
            *CALIBRATION_INPUTS, '--output', TOPP_REPORT, gpu=True, restore_only=REUSE_CALIBRATION)
    else:
        log_event('top-p-diagnostics', 'skipped', LOGS, 'RUN_MASK_TOPP_DIAGNOSTICS=False')
    if TOPP_REPORT.is_file():
        import pandas as pd
        import matplotlib.pyplot as plt
        from IPython.display import display, Image
        top_p_report = json.loads(TOPP_REPORT.read_text(encoding='utf-8'))
        display(pd.DataFrame([{'p':p, **{k:v for k,v in values.items() if k not in ('by_layer','by_source_length','histogram','selected_frames_per_row','retained_teacher_mass','teacher_entropy')}}
            for p,values in top_p_report['candidates'].items()]))
        for p,values in top_p_report['candidates'].items():
            print('p=', p, 'occupancy, retained mass, and selected frames')
            display(pd.DataFrame({key: values[key] for key in ('selected_frames_per_row', 'retained_teacher_mass', 'teacher_entropy')}))
            for group in ('by_layer', 'by_source_length'):
                print(group)
                display(pd.DataFrame([{'Group': name, 'Mean occupancy': stats.get('mean'),
                    'P90 occupancy': stats.get('p90'), 'Mean retained mass': (stats['retained_teacher_mass'] or {}).get('mean'),
                    'Mean selected frames': (stats['selected_frames_per_row'] or {}).get('mean'),
                    'Mean teacher entropy': (stats['teacher_entropy'] or {}).get('mean'),
                    'Valid rows': stats['valid_rows'], 'Invalid rows': stats['invalid_rows']}
                    for name, stats in values[group].items()]))
            plt.plot([(i+.5)/10 for i in range(10)], values['histogram'], label=p)
        plt.xlabel('Mask coverage'); plt.ylabel('Valid teacher rows'); plt.legend()
        plt.savefig(TOPP_ROOT / 'coverage.png'); plt.close()
        display(Image(filename=str(TOPP_ROOT / 'coverage.png')))
    if RUN_ALIGNMENT_CALIBRATION or calibration_saved or REUSE_CALIBRATION:
        if MASK_TOP_P not in (.8,.9,.95):
            raise ValueError('Review the top-p diagnostics and set MASK_TOP_P to 0.8, 0.9, or 0.95.')
        if not TOPP_REPORT.is_file():
            raise ValueError('Run or restore the top-p diagnostics before calibration.')
        previous_options = []
        old_args = ['calibrate', *CALIBRATION_INPUTS, '--diagnostics', TOPP_REPORT,
                    '--mask-top-p', MASK_TOP_P, '--output', PREVIOUS_CALIBRATION]
        module = 'alignment_transfer_s2ut.alignment_calibration'
        expected = [PYTHON, '-m', module, *map(str, old_args)]
        markers = (Path(PERSISTENT) / 'stages' / previous_stage).glob('*/snapshot.json')
        for marker in markers:
            receipt = json.loads(marker.read_text(encoding='utf-8'))
            if 'stage-complete.json' in receipt['files'] and receipt['identity']['command'] == expected:
                run(previous_stage, module, *old_args, restore_only=True)
                previous_options = ['--previous-calibration', PREVIOUS_CALIBRATION]
                break
        if not previous_options:
            log_event('previous-calibration', 'skipped', LOGS, 'No matching completed snapshot')
        run(calibration_stage, module, 'calibrate',
            *CALIBRATION_INPUTS, '--diagnostics', TOPP_REPORT, '--mask-top-p', MASK_TOP_P,
            *previous_options, '--output', CALIBRATION, gpu=True, restore_only=REUSE_CALIBRATION)
    else:
        log_event('alignment-calibration', 'skipped', LOGS, 'RUN_ALIGNMENT_CALIBRATION=False')
    if CALIBRATION.is_file():
        calibrated = json.loads(CALIBRATION.read_text(encoding='utf-8'))
        import pandas as pd
        from IPython.display import display
        print({key:calibrated[key] for key in ('lambda_kl','lambda_mask','tau_kl','tau_mask','selected_top_p','optimizer_updates')})
        if 'gradient_precision' in calibrated:
            print(calibrated['gradient_precision'])
        if 'previous_calibration' in calibrated:
            print('Previous calibration', calibrated['previous_calibration']['coefficients'],
                  'τ matches', calibrated['previous_calibration']['tau_matches'])
        for kind, summary in calibrated['diagnostic_summary'].items():
            print(kind.upper(), 'calibrated gradient diagnostics')
            display(pd.DataFrame({k: summary[k] for k in ('gradient_ratio', 'gradient_cosine')}))
            display(pd.DataFrame([{'Batch': index + 1, **row['gradient_diagnostics'][kind]}
                for index, row in enumerate(calibrated['per_batch'])]))
            weights = summary['layer_weights']
            display(pd.DataFrame({'Mean weight': weights['mean_weights'], 'Worst-layer frequency': weights['worst_layer_frequency']}))
            display(pd.DataFrame({k: weights[k] for k in ('entropy', 'effective_layers', 'max_to_median_weight_ratio')}))
            print({'Maximum layer weight': weights['max_weight'], 'Undefined weight ratios': weights['undefined_weight_ratios']})
else:
    log_event('calibration', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 7. Train the transfer conditions

For charts in Weights & Biases, add the `WANDB_API_KEY` secret in Colab and enable `ENABLE_WANDB`. Your project receives saved training and validation metrics as each Drive snapshot is published.

Select the transfer conditions with `RUN_DISTILLATION`, `RUN_ANNEALED_DISTILLATION`, and `RUN_MASK_DISTILLATION`. Keep `RUN_BASELINE=False` with a completed Baseline. Set `TRANSFER_RUN_MODE='pilot'` to pause at `TRANSFER_PILOT_UPDATES` and collect probes at `TRANSFER_DIAGNOSTIC_STEPS`. The default 20k Pilot includes a measurement after 10k warmup. Every condition keeps the full 400,000-update scheduler and the Baseline recipe.

For a short check on the real corpus, enable one transfer condition and set `TRANSFER_RECOVERY_CHECK='save'` to publish update 2. Open a fresh GPU runtime, keep the same settings, set `TRANSFER_RECOVERY_CHECK='resume'`, and rerun through this section to verify updates 3 and 4. Set the switch back to `'off'` to continue the same run into the Pilot.

Continue with `TRANSFER_RUN_MODE='full'` and the same experiment, data, teacher, and calibration. The same verified Drive snapshots restore the optimizer, scaler, iterator, RNG, and checkpoint selection. Set a new `TRANSFER_EXPERIMENT_ID` for a scientific setting change. Keep `EXPERIMENT_ID` to reuse the original Baseline and corpus.

With `TRANSFER_ONLINE_BLEU=True`, a new full run with empty training and backup locations starts at update 0 and pauses at 1k for evaluation. A published run resumes at its saved update and evaluates at the next milestone. Use the evaluation notebook for older milestones.

In [ ]:
#@title Train the selected transfer conditions

PILOT_RESULTS = []
if not PREPARE_AUDIO_PACKS:
    tracking = prepare_wandb() if ENABLE_WANDB and any(enabled for _, _, enabled in RESEARCH_CONDITIONS) else None
    for method, label, enabled in RESEARCH_CONDITIONS:
        if not enabled:
            log_event(method, 'skipped', LOGS, 'training switch is off')
            continue
        if not CALIBRATION.is_file() or MASK_TOP_P not in (.8, .9, .95):
            raise ValueError('Review top-p diagnostics, choose MASK_TOP_P, and complete calibration first')
        folder = RESEARCH_ROOT / method
        arguments = ['--baseline', str(BASELINE), '--output', str(folder), '--method', method,
            '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'), '--teacher', str(TEACHER_ROOT),
            '--calibration', str(CALIBRATION), '--mask-top-p', str(MASK_TOP_P), '--all-layers', '--resume']
        if method in ('distillation', 'annealed_distillation') and TRANSFER_KL_LAMBDA is not None:
            arguments += ['--kl-lambda', str(TRANSFER_KL_LAMBDA)]
        if method == 'mask_distillation' and TRANSFER_MASK_LAMBDA is not None:
            arguments += ['--mask-lambda', str(TRANSFER_MASK_LAMBDA)]
        plan = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *arguments, name='plan-' + method)
        budget = plan['training_budget']
        print({'condition': label, 'training_budget': budget, 'initialization': plan['initialization'],
               'selection': plan['selection_policy'], 'calibration_hash': plan['transfer']['calibration_hash'],
               'experiment_id': EXPERIMENT_ID, 'transfer_experiment_id': TRANSFER_EXPERIMENT_ID,
               'coefficient_policy': plan['transfer']['coefficient_policy'],
               'checkpoints': str(folder / 'checkpoints')})
        stage = 'research-v3-' + method + SCOPE_SUFFIX
        tracking_args = {'wandb_tracking': (*tracking, plan['contract_hash'], method)} if tracking else {}
        if TRANSFER_RECOVERY_CHECK != 'off':
            check = [PYTHON, str(REPO / 'tools/review_transfer_pilot.py'), 'recovery',
                     '--work', str(folder), '--backup', str(Path(PERSISTENT) / 'stages' / stage),
                     '--method', method, '--phase', TRANSFER_RECOVERY_CHECK, '--baseline', str(BASELINE)]
            check_plan = execute_json(*check, name='recovery-plan-' + method)
            run(stage, 'alignment_transfer_s2ut.benchmark', 'train', *arguments,
                '--stop-after-update', check_plan['target'], gpu=True, **tracking_args)
            report = RESEARCH_ROOT / 'recovery-checks' / method / (TRANSFER_RECOVERY_CHECK + '.json')
            print(execute_json(*check, '--output', str(report), '--report', name='recovery-report-' + method))
            continue
        if TRANSFER_RUN_MODE == 'full':
            if not TRANSFER_ONLINE_BLEU:
                run(stage, 'alignment_transfer_s2ut.benchmark', 'train', *arguments, gpu=True, **tracking_args)
                continue
            from hashlib import sha256
            tool = REPO / 'tools/transfer_bleu.py'
            backup = Path(PERSISTENT) / 'stages' / stage
            output = Path(PERSISTENT) / 'transfer-bleu' / (TRANSFER_EXPERIMENT_ID or 'default')
            # Match the native trainer's exact on-disk serialization and immutable plan.
            plan_bytes = (json.dumps(plan, ensure_ascii=False, indent=2, allow_nan=False) + '\n').encode()
            inventory_args = ['--method', method, '--work', folder, '--backup', backup,
                              '--plan-sha', sha256(plan_bytes).hexdigest()]
            inventory = execute_json(PYTHON, tool, 'inventory', *inventory_args, '--allow-new',
                                     name='bleu-inventory-' + method)
            print(inventory, flush=True)
            for step in inventory['online']:
                stop = ['--stop-after-update', step] if step < budget else []
                evaluation = [PYTHON, tool, 'evaluate', *inventory_args, '--baseline-backup', BACKUP,
                              '--scratch', Path(ROOT) / 'transfer-bleu-work', '--output', output,
                              '--logs', LOGS, '--update', step]
                after = {'after_training': evaluation} if tracking else {}
                run(stage, 'alignment_transfer_s2ut.benchmark', 'train', *arguments, *stop, gpu=True,
                    **tracking_args, **after)
                if not tracking:
                    execute(*evaluation, name=f'bleu-{method}-{step}')
            continue
        if not 0 < TRANSFER_PILOT_UPDATES < budget:
            raise ValueError('TRANSFER_PILOT_UPDATES must be inside the registered training budget')
        if any(type(step) is not int or not 0 < step <= TRANSFER_PILOT_UPDATES for step in TRANSFER_DIAGNOSTIC_STEPS):
            raise ValueError('TRANSFER_DIAGNOSTIC_STEPS must be positive integers within the pilot')
        steps = sorted(set(TRANSFER_DIAGNOSTIC_STEPS) | {TRANSFER_PILOT_UPDATES})
        warmup = plan.get('config', {}).get('lr_scheduler', {}).get('warmup_updates')
        if warmup is not None and not any(step > warmup for step in steps):
            log_event(method, 'warning', LOGS, f'Pilot ends by warmup update {warmup}. Add a later probe to observe post-warmup training.')
        for step in steps:
            probe_stage = f'transfer-probe-{method}-{step}' + SCOPE_SUFFIX
            output = RESEARCH_ROOT / 'pilot-probes' / method / str(step) / 'probe.json'
            saved = any((Path(PERSISTENT) / 'stages' / probe_stage).glob('*/snapshot.json'))
            if step == TRANSFER_PILOT_UPDATES or not saved:
                run(stage, 'alignment_transfer_s2ut.benchmark', 'train', *arguments, '--stop-after-update', step, gpu=True, **tracking_args)
                session = json.loads((folder / 'native-session.json').read_text(encoding='utf-8'))
                if session['durable_updates'] != step and not saved:
                    log_event(probe_stage, 'skipped', LOGS, f"checkpoint already advanced to {session['durable_updates']}")
                    continue
            run(probe_stage, 'alignment_transfer_s2ut.transfer_pilot', 'probe', '--run', folder,
                '--update', step, '--calibration', CALIBRATION, '--output', output, gpu=True, restore_only=saved)
            report = json.loads(output.read_text(encoding='utf-8'))
            if report['contract_hash'] != plan['contract_hash'] or report['step'] != step:
                raise ValueError('Saved pilot probe differs from the requested contract or update')
            PILOT_RESULTS.append(report)
else:
    log_event('transfer-training', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

In [ ]:
#@title Register the completed transfer checkpoints

if not PREPARE_AUDIO_PACKS and NEED_TEACHER:
    conditions = [('baseline', 'Baseline', BASELINE)]
    for method, label, enabled in RESEARCH_CONDITIONS:
        folder = RESEARCH_ROOT / method
        if enabled and TRANSFER_RUN_MODE == 'full':
            record = RESEARCH_ROOT / 'registrations' / (method + '.json')
            run('research-v3-register-' + method + TRANSFER_SUFFIX, 'alignment_transfer_s2ut.benchmark', 'register-transfer',
                '--run', folder, '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
            conditions.append((method,label,record))
        elif enabled:
            log_event(method + '-registration', 'skipped', LOGS, 'pilot checkpoints are for diagnostics')
else:
    log_event('transfer-registration', 'skipped', LOGS, 'stage disabled or CPU audio preparation')

## 8. Training diagnostics and evaluation

Review Pilot loss, learning rate, layer weights, gradient ratio and cosine, and the fixed dev diagnostics. The Baseline column uses its dev-selected checkpoint from the completed 400k run. Use it as an Attention reference at a different training stage. Read gradient ratio and cosine alongside task loss and Attention KL. Optionally set `EVALUATE_TRANSFER_PILOT_DEV=True` for full dev UER and `TRANSFER_PILOT_ASR_BLEU=True` for full dev ASR-BLEU. Pilot scores describe the current checkpoints. Compare completed 400k models in [evaluate.ipynb](evaluate.ipynb).

Set `RUN_ALIGNMENT_DIAGNOSTICS=True` for the existing completed-model layer diagnostics and visualizations.

In [ ]:
#@title Compare Pilot metrics

if globals().get('PILOT_RESULTS'):
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display, Image

    baseline_dev = RESEARCH_ROOT / 'pilot-probes' / 'baseline' / 'dev-alignment.json'
    run('transfer-probe-baseline' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.transfer_pilot', 'baseline-dev',
        '--baseline', BASELINE, '--manifest', MANIFEST, '--units', Path(ROOT) / 'units', '--teacher', TEACHER_ROOT,
        '--calibration', CALIBRATION, '--output', baseline_dev, gpu=True)
    baseline_metrics = json.loads(baseline_dev.read_text(encoding='utf-8'))
    rows = []
    for report in PILOT_RESULTS:
        dev = report['dev']
        if dev['pair_ids'] != baseline_metrics['pair_ids']:
            raise ValueError('Pilot and Baseline dev diagnostics use different samples')
        rows.append({'Condition': report['method'], 'Update': report['step'],
            'λ': report['lambda_t'], 'τ': report['tau'], 'Gradient ratio': report['gradient_ratio'],
            'Gradient cosine': report['gradient_cosine'], 'Probe seconds': report['gradient_seconds'],
            'Peak CUDA bytes': report['peak_cuda_bytes'], 'Warnings': ', '.join(report['warnings']),
            'Dev task loss (fixed subset)': dev['dev_task_loss'], 'Full dev unit loss': report['full_dev_unit_loss'],
            'Dev KL': dev['kl_mean'], 'Mask-region mass': sum(dev['teacher_region_mass']) / len(dev['teacher_region_mass']),
            'Valid attention rows': sum(dev['valid_rows'])})
        print(report['method'], report['step'])
        display(pd.DataFrame({'KL': dev['per_layer_kl'], 'Baseline KL': baseline_metrics['per_layer_kl'],
            'Mask-region mass': dev['teacher_region_mass'], 'Baseline mass': baseline_metrics['teacher_region_mass'],
            'Worst-layer weight': dev['worst_layer_weight_distribution'], 'Valid rows': dev['valid_rows']}))
    baseline_updates = json.loads(BASELINE.read_text(encoding='utf-8'))['checkpoint_updates']
    rows.append({'Condition': 'Baseline (completed run reference)', 'Update': baseline_updates, 'Dev task loss (fixed subset)': baseline_metrics['dev_task_loss'],
                 'Dev KL': baseline_metrics['kl_mean'],
                 'Mask-region mass': sum(baseline_metrics['teacher_region_mass']) / len(baseline_metrics['teacher_region_mass']),
                 'Valid attention rows': sum(baseline_metrics['valid_rows'])})
    display(pd.DataFrame(rows))
    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, constrained_layout=True)
    folders = []
    for method in dict.fromkeys(r['method'] for r in PILOT_RESULTS):
        folders.append(RESEARCH_ROOT / method)
        observed = [r for r in PILOT_RESULTS if r['method'] == method]
        for axis, key, label in zip(axes, ('gradient_ratio', 'gradient_cosine'), ('Gradient norm ratio', 'Gradient cosine')):
            axis.plot([r['step'] for r in observed], [r[key] for r in observed], marker='o', label=method)
            axis.set_ylabel(label)
            axis.legend()
    axes[-1].set_xlabel('Completed updates')
    display(fig)
    plt.close(fig)
    curves = RESEARCH_ROOT / 'pilot-curves'
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.alignment_report', 'curves', '--runs', *map(str, folders),
            '--output', str(curves), name='transfer-pilot-curves')
    for folder in folders:
        latest = json.loads((folder / 'alignment-train.jsonl').read_text(encoding='utf-8').splitlines()[-1])
        print(folder.name, {key: latest.get(key) for key in ('step', 'alignment/task_loss', 'alignment/loss',
              'train/unit_ce_nats', 'train/optimization_loss_nats', 'train/learning_rate', 'train/seconds')})
        display(Image(filename=str(curves / (folder.name + '.png'))))
    if EVALUATE_TRANSFER_PILOT_DEV:
        score_mode, flags = 'units', []
        if TRANSFER_PILOT_ASR_BLEU:
            run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
            score_mode = 'asr'
            flags = ['--vocoder', 'configs/s2ut/vocoder.yaml', '--asr', 'configs/experiments/evaluation.yaml']
        checkpoints = [('Baseline', ['--baseline', BASELINE])]
        for folder in folders:
            state = json.loads((folder / 'native-session.json').read_text(encoding='utf-8'))
            if state['status'] == 'PILOT_PAUSED' and state['durable_updates'] == TRANSFER_PILOT_UPDATES:
                checkpoints.append((folder.name, ['--run', folder]))
        translation = []
        for label, checkpoint_args in checkpoints:
            output = RESEARCH_ROOT / 'pilot-translation' / str(TRANSFER_PILOT_UPDATES) / label / score_mode / 'result.json'
            run(f'transfer-pilot-dev-{label}-{TRANSFER_PILOT_UPDATES}-{score_mode}' + TRANSFER_SUFFIX,
                'alignment_transfer_s2ut.transfer_pilot', 'translate', *checkpoint_args, '--manifest', MANIFEST,
                '--units', Path(ROOT) / 'units', '--output', output, *flags, gpu=True)
            result = json.loads(output.read_text(encoding='utf-8'))
            translation.append({'Condition': label, 'Dev UER': result.get('unit_error_rate'), 'Dev ASR-BLEU': result.get('asr_bleu')})
        display(pd.DataFrame(translation))
else:
    log_event('transfer-pilot-diagnostics', 'skipped', LOGS, 'no pilot probes to display')

In [ ]:
#@title Inspect attention projection by layer

if not PREPARE_AUDIO_PACKS and RUN_ALIGNMENT_DIAGNOSTICS:
    # Run teacher-forced dev diagnostics separately from generation.
    if globals().get('conditions') and CALIBRATION.is_file():
        import pandas as pd
        from IPython.display import display, Image
        completed = []
        for name, label, record in conditions:
            output = RESEARCH_ROOT / 'alignment-diagnostics' / name / 'diagnostics.json'
            run('research-v3-alignment-' + name + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_report', 'diagnose',
                '--baseline', BASELINE, '--record', record, '--manifest', MANIFEST, '--units', Path(ROOT) / 'units',
                '--teacher', TEACHER_ROOT, '--calibration', CALIBRATION, '--samples', 32, '--output', output, gpu=True)
            result = json.loads(output.read_text(encoding='utf-8'))
            print(label, 'KL mean/worst:', result['kl_mean'], result['kl_worst'])
            display(pd.DataFrame({'KL':result['per_layer_kl'], 'Mask loss':result['per_layer_mask_loss'],
                'Region mass':result['teacher_region_mass'], 'Worst-layer weight':result['worst_layer_weight_distribution']}))
            if name != 'baseline':
                dev_pair = next(json.loads(line)['pair_id'] for line in MANIFEST.read_text(encoding='utf-8').splitlines()
                    if line.strip() and json.loads(line)['split']=='dev')
                visual = RESEARCH_ROOT / 'attention-visualization' / name
                run('research-v3-visualize-' + name + SCOPE_SUFFIX, 'alignment_transfer_s2ut.visualize_attention',
                    '--manifest', MANIFEST, '--checkpoint', RESEARCH_ROOT / name / 'checkpoints/checkpoint_best.pt',
                    '--units', Path(ROOT) / 'units', '--teacher', TEACHER_ROOT, '--pair-id', dev_pair,
                    '--device', 'cuda', '--output', visual, '--overwrite', gpu=True)
                for path in visual.glob('*.png'):
                    display(Image(filename=str(path)))
            if name != 'baseline':
                completed.append(RESEARCH_ROOT / name)
        if completed:
            curves_root = RESEARCH_ROOT / 'curves'
            run('research-v3-curves' + SCOPE_SUFFIX, 'alignment_transfer_s2ut.alignment_report', 'curves',
                '--runs', *completed, '--output', curves_root)
            for folder in completed:
                display(Image(filename=str(curves_root / (folder.name + '.png'))))
    else:
        log_event('layer-diagnostics', 'skipped', LOGS, 'complete calibration and common evaluation first')
else:
    log_event('layer-diagnostics', 'skipped', LOGS, 'stage disabled or CPU audio preparation')